# OzzyVision-Lab
### LTX-2.5 & MiniMax H3 — Google Colab A100

Bu notebook, **LTX-2.5 Distilled Q8 GGUF** ve **MiniMax H3 Omni Ref2VA Safetensors (Resmi Comfy-Org)** modellerini kullanarak Google Colab A100 GPU üzerinde ComfyUI headless motoru, FastAPI backend, modern React stüdyo arayüzü ve Claude uyumlu Remote MCP sunucusunu tek tıkla ayağa kaldırır.

📁 **Kalıcı Depolama:** Modeller, videolar ve ayarlar Google Drive'ınızdaki `MyDrive/OzzyVision-Lab` klasöründe saklanır. Modeller bir kez indirilir, Colab her açıldığında kontrol edilerek tekrar indirilmez.

🔑 **API Anahtarları (Colab Secrets):**
- `HF_TOKEN`: Hugging Face modellerini hızlı ve kota engelsiz indirmek için.
- `NGROK_AUTHTOKEN`: Web Stüdyosu arayüzünü tarayıcınızda açmak ve Claude MCP bağlantısı için.

**Önemli:** Runtime türünüzün **A100 GPU** olduğundan emin olun (`Runtime -> Change runtime type -> A100 GPU`).

## 1. Adım: GPU Kontrolü & Donanım Doğrulama

In [ ]:
# 1. GPU ve VRAM Kontrolü
!nvidia-smi
import torch
print("CUDA Kullanılabilir mi:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Cihazı:", torch.cuda.get_device_name(0))
    print("Toplam VRAM:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1), "GB")
else:
    print("UYARI: CUDA tespit edilemedi! Runtime GPU seçili mi kontrol edin.")

# 2. 16GB SWAP (sistem RAM'i dolarsa çökme yerine yavaşlama)
!if [ ! -f /swapfile ]; then fallocate -l 16G /swapfile 2>/dev/null && chmod 600 /swapfile && mkswap /swapfile 2>/dev/null && swapon /swapfile 2>/dev/null && echo "✅ 16GB SWAP takas belleği devrede!"; else echo "ℹ️ SWAP takas belleği zaten aktif."; fi
!free -h


## 2. Adım: Google Drive Bağlama & OzzyVision-Lab Klasörleri
Google Drive `OzzyVision-Lab` klasörünü bağlar. Proje kodları ve modeller bu klasörde kalıcı olarak saklanır.


In [ ]:
import os
from google.colab import drive

# 1. Drive Mount
drive.mount('/content/drive')

# Otomatik Taşıma: Eğer Drive'da eski LTX-Studio klasörü varsa OzzyVision-Lab olarak güncelle
old_drive = '/content/drive/MyDrive/LTX-Studio'
new_drive = '/content/drive/MyDrive/OzzyVision-Lab'
if os.path.exists(old_drive) and not os.path.exists(new_drive):
    print(f'📦 Mevcut {old_drive} klasörü {new_drive} olarak güncelleniyor...')
    os.rename(old_drive, new_drive)

# 2. OzzyVision-Lab Proje Yolları
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/OzzyVision-Lab"
DRIVE_MODELS = f"{DRIVE_PROJECT_ROOT}/models"
DRIVE_JOBS = f"{DRIVE_PROJECT_ROOT}/jobs"
DRIVE_GALLERY = f"{DRIVE_PROJECT_ROOT}/gallery"
DRIVE_ASSETS = f"{DRIVE_PROJECT_ROOT}/assets"
DRIVE_LORAS = f"{DRIVE_MODELS}/loras"

# Dizinleri oluştur
for d in [DRIVE_PROJECT_ROOT, DRIVE_MODELS, DRIVE_JOBS, DRIVE_GALLERY, DRIVE_ASSETS, DRIVE_LORAS]:
    os.makedirs(d, exist_ok=True)

# Model alt dizinleri
os.makedirs(f"{DRIVE_MODELS}/diffusion_models", exist_ok=True)
os.makedirs(f"{DRIVE_MODELS}/text_encoders", exist_ok=True)
os.makedirs(f"{DRIVE_MODELS}/vae", exist_ok=True)
os.makedirs(f"{DRIVE_MODELS}/latent_upscale_models", exist_ok=True)

# 3. PROJE KODLARI SENKRONİZASYONU
# Kod her oturumda Drive ile senkronlanır. Kaynak olarak:
#   a) PROJECT_GIT_URL (önerilen), b) /content/OzzyVision-Lab-repo, c) Drive'a elle yükleme.
PROJECT_GIT_URL = "https://github.com/OzzyD07/OzzyVision-Lab.git"

code_source = None

if PROJECT_GIT_URL:
    src_dir = "/content/_ozzy_src"
    if os.path.exists(f"{src_dir}/.git"):
        print("🔄 Depo güncelleniyor (git pull)...")
        !cd {src_dir} && git pull --ff-only
    else:
        print("⬇️ Depo klonlanıyor...")
        !rm -rf {src_dir} && git clone --depth 1 {PROJECT_GIT_URL} {src_dir}
    if os.path.exists(f"{src_dir}/app"):
        code_source = src_dir

if not code_source:
    for cand in ["/content/OzzyVision-Lab-repo", "/content/OzzyVision-Lab_repo", "/content/VideoAi", "/content/_ozzy_src"]:
        if os.path.exists(f"{cand}/app"):
            code_source = cand
            break

if code_source:
    print(f"📁 Kodlar {code_source} dizininden Google Drive'a aktarılıyor...")
    # Yalnızca kod klasörleri; jobs/gallery/assets/models üretilen veridir, dokunulmaz.
    for _sub in ["app", "config", "tests", "colab"]:
        if os.path.exists(f"{code_source}/{_sub}"):
            !mkdir -p {DRIVE_PROJECT_ROOT}/{_sub}
            !cp -ru {code_source}/{_sub}/. {DRIVE_PROJECT_ROOT}/{_sub}/ 2>/dev/null || true
    for _f in ["requirements.txt", "README.md"]:
        if os.path.exists(f"{code_source}/{_f}"):
            !cp -u {code_source}/{_f} {DRIVE_PROJECT_ROOT}/ 2>/dev/null || true
else:
    print("ℹ️ Yerel kod kaynağı bulunamadı; Drive'daki mevcut kod kullanılacak.")

# Bayat .pyc önbelleğini temizle
!find {DRIVE_PROJECT_ROOT} -name "__pycache__" -type d -exec rm -rf {{}} + 2>/dev/null || true

# Proje kodlarını /content/OzzyVision-Lab dizinine bağla
!rm -f /content/OzzyVision-Lab
!ln -sfn /content/drive/MyDrive/OzzyVision-Lab /content/OzzyVision-Lab

# 4. KOD SÜRÜM DOĞRULAMASI
_settings_path = f"{DRIVE_PROJECT_ROOT}/config/settings.py"
if os.path.exists(_settings_path):
    _txt = open(_settings_path, encoding="utf-8").read()
    if "FREE_VRAM_ON_MODEL_SWITCH" in _txt:
        print("✅ Kod sürümü GÜNCEL (VRAM/OOM düzeltmeleri mevcut).")
    else:
        print("=" * 68)
        print("⚠️  DİKKAT: Drive'daki kod ESKİ sürüm!")
        print("   LoRA kullanırken 'CUDA out of memory' hatası devam edecektir.")
        print("   PROJECT_GIT_URL'i doldurup bu hücreyi tekrar çalıştırın veya")
        print("   güncel dosyaları Drive'daki OzzyVision-Lab klasörüne yükleyin.")
        print("=" * 68)
else:
    print("⚠️ config/settings.py bulunamadı! Proje kodları Drive'a yüklenmemiş olabilir.")

print("✅ Google Drive bağlandı ve OzzyVision-Lab dizinleri hazır:", DRIVE_PROJECT_ROOT)


## 3. Adım: ComfyUI & Gerekli Paketlerin Kurulumu
Inference motoru olan ComfyUI ve tüm bağımlılıkları kurulur; GGUF loader ve VideoHelperSuite eklentileri yüklenir.

In [ ]:
import os

# 1. ComfyUI İndirme
%cd /content
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git

# 2. Custom Node'lar (ComfyUI-GGUF, VHS ve H3-Multishot)
%cd /content/ComfyUI/custom_nodes
if not os.path.exists('ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git
if not os.path.exists('ComfyUI-VideoHelperSuite'):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
if not os.path.exists('ComfyUI-LTXVideo'):
    !git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git || true
if not os.path.exists('ComfyUI-H3-Multishot'):
    !git clone https://github.com/jlucasmcrell/ComfyUI-H3-Multishot.git || true

# 3. ComfyUI ve Eklenti Bağımlılıkları
print("📦 ComfyUI ve sistem bağımlılıkları yükleniyor...")
!pip install -q -r /content/ComfyUI/requirements.txt
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt imageio-ffmpeg opencv-python
!pip install -q fastapi "uvicorn[standard]" pyngrok pydantic python-multipart gguf huggingface_hub websockets requests

# Proje bağımlılıkları (requirements.txt) - Drive'daki kod kopyasından
if os.path.exists('/content/OzzyVision-Lab/requirements.txt'):
    !pip install -q -r /content/OzzyVision-Lab/requirements.txt

# LTX-Video özel düğümlerinin bağımlılıkları (varsa)
for _req in ['/content/ComfyUI/custom_nodes/ComfyUI-LTXVideo/requirements.txt',
             '/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt']:
    if os.path.exists(_req):
        !pip install -q -r {_req}

# 4. REACT ARAYÜZÜNÜN DERLENMESİ
# FastAPI arayüzü app/frontend/dist içinden servis eder.
FRONTEND_DIR = "/content/OzzyVision-Lab/app/frontend"
DIST_INDEX = f"{FRONTEND_DIR}/dist/index.html"

def _frontend_is_stale():
    """dist yoksa veya kaynak dosyalar dist'ten yeniyse yeniden derleme gerekir."""
    if not os.path.exists(DIST_INDEX):
        return True
    dist_mtime = os.path.getmtime(DIST_INDEX)
    for root, _, files in os.walk(f"{FRONTEND_DIR}/src"):
        for fn in files:
            if os.path.getmtime(os.path.join(root, fn)) > dist_mtime:
                return True
    for fn in ("index.html", "vite.config.js", "package.json"):
        fp = os.path.join(FRONTEND_DIR, fn)
        if os.path.exists(fp) and os.path.getmtime(fp) > dist_mtime:
            return True
    return False

if not os.path.exists(f"{FRONTEND_DIR}/package.json"):
    print("⚠️ app/frontend bulunamadı; arayüz derlenemiyor.")
elif not _frontend_is_stale():
    print("✅ React stüdyo arayüzü zaten güncel (dist hazır), derleme atlandı.")
else:
    print("🎨 React stüdyo arayüzü derleniyor (ilk seferde ~1-2 dk)...")
    # Drive'da npm yavaş: yerel SSD'de derleyip sonucu kopyala
    !rm -rf /content/_frontend_build && mkdir -p /content/_frontend_build
    !cp -r {FRONTEND_DIR}/src {FRONTEND_DIR}/index.html {FRONTEND_DIR}/vite.config.js {FRONTEND_DIR}/package.json /content/_frontend_build/ 2>/dev/null
    !cd /content/_frontend_build && npm install --silent --no-audit --no-fund && npm run build
    if os.path.exists("/content/_frontend_build/dist/index.html"):
        !rm -rf {FRONTEND_DIR}/dist && mkdir -p {FRONTEND_DIR}/dist
        !cp -r /content/_frontend_build/dist/. {FRONTEND_DIR}/dist/
        print("✅ Stüdyo arayüzü derlendi ve Drive'a kaydedildi.")
    else:
        print("❌ Arayüz derlenemedi! Web stüdyosu yerine API mesajı görünecektir.")

print("✅ Tüm bağımlılıklar başarıyla kuruldu!")


## 4. Adım: Model Dosyaları & Drive Kalıcı Depo Kontrolü (LTX-2.5 & MiniMax H3)
Modeller doğrudan Google Drive'ınızdaki `OzzyVision-Lab/models` içine indirilir. Eğer dosyalar Drive'da zaten mevcutsa **tekrar indirilmez**, hemen ComfyUI'ye bağlanır.

- **LTX-2.5 Distilled:** Hafif & ultra hızlı (~13B param, 8 adım).
- **MiniMax H3 Omni Ref2VA:** 9 Görsel, 3 Video, 3 Ses referansı ve doğal 32 kHz stereo sesli konuşma üretimi (~33B param).


In [ ]:
# ==========================================
# 4. MODEL KONTROLÜ VE İNDİRME SİSTEMİ
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download, login

# Colab Secrets'tan HF_TOKEN al (varsa)
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ""
except Exception:
    HF_TOKEN = os.getenv('HF_TOKEN', "")

if HF_TOKEN:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("🔑 Hugging Face erişim jetonu (HF_TOKEN) başarıyla bağlandı.")
    except Exception as e:
        print(f"⚠️ HF giriş uyarısı: {e}")
else:
    print("ℹ️ HF_TOKEN bulunamadı. Açık modeller anonim olarak indirilecektir.")

# ==========================================
# İNDİRİLECEK MODELLER LİSTESİ
# ==========================================
# Her model için birden fazla kaynak; ilki başarısız olursa sıradaki denenir.
#
# LTX-2.5 NOTU: Resmi depo "Lightricks/LTX-2.5" LİSANS ONAYI ister.
# https://huggingface.co/Lightricks/LTX-2.5 adresine gidip lisansı kabul edin
# ve HF_TOKEN'ı Colab Secrets'a ekleyin. Onaylamazsanız açık ayna depo kullanılır.
#
# LTX_QUANT: "int8" -> resmi Comfy int8 safetensors (~18.7 GB, en iyi kalite)
#            "gguf" -> Q8_0 GGUF (~23.6 GB) veya Q5_K_M (~15 GB) ile düşük VRAM
LTX_QUANT = "int8"   # "int8" | "gguf"

_LTX_OFFICIAL = "Lightricks/LTX-2.5"
_LTX_MIRROR = "comfyicu/LTX-2.5"
_LTX_GGUF = "vantagewithai/LTX-2.5-GGUF"
_MMX = "Comfy-Org/MiniMax-H3"

if LTX_QUANT == "gguf":
    _ltx_transformer = {
        "filename": "ltx-2.5-22b-distilled-transformer-Q8_0.gguf",
        "sources": [
            {"repo": _LTX_GGUF, "file": "distilled/ltx-2.5-22b-distilled-transformer-Q8_0.gguf"},
        ],
    }
else:
    _ltx_transformer = {
        "filename": "ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors",
        "sources": [
            {"repo": _LTX_OFFICIAL, "file": "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"},
            {"repo": _LTX_MIRROR, "file": "diffusion_models/ltx-2.5-22b-distilled-transformer-comfy-int8-convrot.safetensors"},
        ],
    }

models_to_check = [
    # --- 1. LTX-2.5 Distilled (22B) Modelleri ---
    {
        "name": "LTX-2.5 Distilled 22B Transformer",
        "category": "LTX-2.5 Diffusion",
        "subdir": "diffusion_models",
        "filename": _ltx_transformer["filename"],
        "sources": _ltx_transformer["sources"],
    },
    {
        "name": "Gemma 4 12B Text Encoder (Comfy Int8)",
        "category": "LTX-2.5 CLIP",
        "subdir": "text_encoders",
        "filename": "gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors",
        "sources": [
            {"repo": _LTX_OFFICIAL, "file": "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"},
            {"repo": _LTX_MIRROR, "file": "text_encoders/gemma4-12b-with-proj-ltx-2.5-comfy-int8-convrot.safetensors"},
        ],
    },
    {
        "name": "LTX-2.5 Video VAE (BF16)",
        "category": "LTX-2.5 VAE",
        "subdir": "vae",
        "filename": "ltx-2.5-video-vae-bf16.safetensors",
        "sources": [
            {"repo": _LTX_OFFICIAL, "file": "vae/ltx-2.5-video-vae-bf16.safetensors"},
            {"repo": _LTX_MIRROR, "file": "vae/ltx-2.5-video-vae-bf16.safetensors"},
        ],
    },
    {
        "name": "LTX-2.5 Audio VAE (BF16)",
        "category": "LTX-2.5 Audio VAE",
        "subdir": "vae",
        "filename": "ltx-2.5-audio-vae-bf16.safetensors",
        "sources": [
            {"repo": _LTX_OFFICIAL, "file": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
            {"repo": _LTX_MIRROR, "file": "vae/ltx-2.5-audio-vae-bf16.safetensors"},
        ],
    },
    {
        "name": "LTX-2.5 Spatial Upscaler x2 (Opsiyonel)",
        "category": "LTX-2.5 Upscaler",
        "subdir": "latent_upscale_models",
        "filename": "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors",
        "optional": True,
        "sources": [
            {"repo": _LTX_OFFICIAL, "file": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
            {"repo": _LTX_MIRROR, "file": "latent_upscale_models/ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"},
        ],
    },
    # --- 2. MiniMax H3 Omni Ref2VA Modelleri (Resmi Comfy-Org) ---
    {
        "name": "MiniMax H3 Ref2VA Diffusion (FP8 Scaled)",
        "category": "MiniMax H3 Diffusion",
        "subdir": "diffusion_models",
        "filename": "minimax_h3_ref2va_pruned_fp8_scaled.safetensors",
        "sources": [{"repo": _MMX, "file": "diffusion_models/minimax_h3_ref2va_pruned_fp8_scaled.safetensors"}],
    },
    {
        "name": "Qwen3-VL 32B Text Encoder (NVFP4 AWQ)",
        "category": "MiniMax H3 Text Encoder",
        "subdir": "text_encoders",
        "filename": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
        "sources": [{"repo": _MMX, "file": "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"}],
    },
    {
        "name": "MiniMax H3 Video VAE (FP16)",
        "category": "MiniMax H3 Video VAE",
        "subdir": "vae",
        "filename": "minimax_h3_video_vae_fp16.safetensors",
        "sources": [{"repo": _MMX, "file": "vae/minimax_h3_video_vae_fp16.safetensors"}],
    },
    {
        "name": "MiniMax H3 Audio VAE (FP32)",
        "category": "MiniMax H3 Audio VAE",
        "subdir": "vae",
        "filename": "minimax_h3_audio_vae_fp32.safetensors",
        "sources": [{"repo": _MMX, "file": "vae/minimax_h3_audio_vae_fp32.safetensors"}],
    },
]

# ==========================================
# İNDİRME DÖNGÜSÜ (MEVCUT DOSYALAR ATLANIR)
# ==========================================
print("")
print("🔍 Model dosyaları Google Drive üzerinde kontrol ediliyor...")

failed_models = []
LTX_MODEL_FILENAME = _ltx_transformer["filename"]

for m in models_to_check:
    target_dir = f"{DRIVE_MODELS}/{m['subdir']}"
    target_path = f"{target_dir}/{m['filename']}"
    os.makedirs(target_dir, exist_ok=True)

    if os.path.exists(target_path) and os.path.getsize(target_path) > 1024 * 1024:
        size_gb = round(os.path.getsize(target_path) / (1024**3), 2)
        print(f"   ✅ [MEVCUT] [{m['category']}] {m['name']} ({size_gb} GB) Drive'da hazır, indirme ATLANDI.")
        continue

    last_error = None
    downloaded_ok = False

    for src in m["sources"]:
        print(f"   ⬇️  [{m['category']}] {m['name']} indiriliyor... (kaynak: {src['repo']})")
        try:
            downloaded_path = hf_hub_download(
                repo_id=src["repo"],
                filename=src["file"],
                local_dir=target_dir,
                token=HF_TOKEN if HF_TOKEN else None,
            )
            if os.path.abspath(downloaded_path) != os.path.abspath(target_path):
                shutil.move(downloaded_path, target_path)
            size_gb = round(os.path.getsize(target_path) / (1024**3), 2)
            print(f"   🎉 {m['name']} ({size_gb} GB) indirildi ve Drive'a kaydedildi!")
            downloaded_ok = True
            break
        except Exception as err:
            last_error = err
            emsg = str(err).lower()
            if ("gated" in emsg) or ("401" in emsg) or ("403" in emsg) or ("awaiting" in emsg):
                print(f"      🔒 {src['repo']} lisans onayı gerektiriyor.")
                print(f"         https://huggingface.co/{src['repo']} adresinden lisansı kabul edip")
                print("         HF_TOKEN'ı Colab Secrets'a ekleyin. Yedek kaynak deneniyor...")
            elif "404" in emsg or "entry not found" in emsg:
                print(f"      ⚠️  {src['repo']} içinde dosya bulunamadı. Yedek kaynak deneniyor...")
            else:
                print(f"      ⚠️  Hata: {str(err)[:140]}")

    if not downloaded_ok:
        # Boş/yarım kalan dosyayı temizle
        if os.path.exists(target_path) and os.path.getsize(target_path) < 1024 * 1024:
            try:
                os.remove(target_path)
            except OSError:
                pass
        if m.get("optional"):
            print(f"   ↷ [ATLANDI] {m['name']} opsiyonel; olmadan devam ediliyor.")
        else:
            print(f"   ❌ {m['name']} hiçbir kaynaktan indirilemedi.")
            failed_models.append((m["name"], str(last_error)))

# Eksik model raporu
print("")
if failed_models:
    print("⚠️  İNDİRİLEMEYEN MODELLER:")
    for _name, _err in failed_models:
        print(f"   • {_name}: {_err[:160]}")
    print("   Bu modelleri kullanan motor çalışmayacaktır; diğerleri kullanılabilir.")
else:
    print("✅ Tüm model dosyaları hazır.")

# ==========================================
# COMFYUI DİZİNLERİNE BAĞLAMA & HIZLANDIRMA MODU
# ==========================================
# ⚡ HIZ VE PERFORMANS SEÇENEKLERİ:
# "instant"      -> 0 SANİYE! (TAVSİYE EDİLİR) Modeller doğrudan Drive'dan symlink ile bağlanır.
# "fast_minimax" -> Yalnızca MiniMax H3 modellerini yerel NVMe SSD'ye kopyalar (~3 dk).
# "full_cache"   -> Tüm modelleri yerel SSD'ye kopyalar (~10 dk).

CACHE_MODE = "instant"  # "instant" | "fast_minimax" | "full_cache"

# LoRA dizinini komple symlink'le: Drive'a sonradan eklenen LoRA'lar da görünür.
!mkdir -p /content/drive/MyDrive/OzzyVision-Lab/models/loras
import os as _o, shutil as _sh
_comfy_loras = '/content/ComfyUI/models/loras'
_drive_loras = '/content/drive/MyDrive/OzzyVision-Lab/models/loras'
if _o.path.islink(_comfy_loras):
    _o.unlink(_comfy_loras)
elif _o.path.isdir(_comfy_loras):
    # Yerelde duran LoRA'ları kaybetmemek için önce Drive'a taşı
    for _f in _o.listdir(_comfy_loras):
        _srcf = _o.path.join(_comfy_loras, _f)
        if _o.path.isfile(_srcf) and not _o.path.exists(_o.path.join(_drive_loras, _f)):
            _sh.move(_srcf, _o.path.join(_drive_loras, _f))
    _sh.rmtree(_comfy_loras, ignore_errors=True)
_o.symlink(_drive_loras, _comfy_loras)
print(f"🔗 LoRA dizini bağlandı: {_comfy_loras} -> {_drive_loras}")
!mkdir -p /content/ComfyUI/models/diffusion_models /content/ComfyUI/models/unet /content/ComfyUI/models/text_encoders /content/ComfyUI/models/clip /content/ComfyUI/models/vae /content/ComfyUI/models/latent_upscale_models
!ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/vae/* /content/ComfyUI/models/vae/ 2>/dev/null || true
!ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/latent_upscale_models/* /content/ComfyUI/models/latent_upscale_models/ 2>/dev/null || true

free_gb = round(shutil.disk_usage("/content").free / (1024**3), 1)
print(f"⚡ Colab Yerel SSD Boş Alanı: {free_gb} GB | Seçilen Mod: [{CACHE_MODE}]")

if CACHE_MODE == "instant":
    print("⚡ [INSTANT MOD]: Modeller doğrudan Google Drive symlink ile bağlandı (Sıfır bekleme süresi, anında hazır!).")
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/diffusion_models/* /content/ComfyUI/models/diffusion_models/ 2>/dev/null || true
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/diffusion_models/* /content/ComfyUI/models/unet/ 2>/dev/null || true
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/text_encoders/* /content/ComfyUI/models/text_encoders/ 2>/dev/null || true
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/text_encoders/* /content/ComfyUI/models/clip/ 2>/dev/null || true
elif CACHE_MODE == "fast_minimax" and free_gb > 35:
    print("🚀 [HIZLI PARALEL MOD]: Yalnızca MiniMax H3 modelleri paralel çift kanalla kopyalanıyor...")
    ! (cp -u /content/drive/MyDrive/OzzyVision-Lab/models/diffusion_models/*minimax* /content/ComfyUI/models/diffusion_models/ 2>/dev/null & \
       cp -u /content/drive/MyDrive/OzzyVision-Lab/models/text_encoders/*minimax* /content/ComfyUI/models/text_encoders/ 2>/dev/null & \
       wait) && echo "✅ MiniMax H3 modelleri paralel başarıyla yerel SSD'ye aktarıldı!"
    !ln -sf /content/ComfyUI/models/diffusion_models/* /content/ComfyUI/models/unet/ 2>/dev/null || true
    !ln -sf /content/ComfyUI/models/text_encoders/* /content/ComfyUI/models/clip/ 2>/dev/null || true
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/diffusion_models/* /content/ComfyUI/models/diffusion_models/ 2>/dev/null || true
    !ln -sf /content/drive/MyDrive/OzzyVision-Lab/models/text_encoders/* /content/ComfyUI/models/text_encoders/ 2>/dev/null || true
else:
    print("📦 [TAM KOPYALAMA MODU]: Tüm modeller yerel SSD'ye kopyalanıyor...")
    ! (rsync -ah --info=progress2 /content/drive/MyDrive/OzzyVision-Lab/models/diffusion_models/* /content/ComfyUI/models/diffusion_models/ 2>/dev/null & \
       rsync -ah --info=progress2 /content/drive/MyDrive/OzzyVision-Lab/models/text_encoders/* /content/ComfyUI/models/text_encoders/ 2>/dev/null & \
       wait) || true
    !ln -sf /content/ComfyUI/models/diffusion_models/* /content/ComfyUI/models/unet/ 2>/dev/null || true
    !ln -sf /content/ComfyUI/models/text_encoders/* /content/ComfyUI/models/clip/ 2>/dev/null || true

print("\n🚀 Tüm modeller başarıyla ComfyUI motoruna bağlandı ve kullanıma hazır!")


## 5. Adım: FAZ-01 Bağımsız Doğrulama Testi (Fidelity Test - Opsiyonel)
Web arayüzünü açmadan önce tek bir referans görsel ve prompt ile Q8 GGUF modelinin video ürettiğini doğrulamak için bu adımı çalıştırabilirsiniz.

In [ ]:
import subprocess, time, urllib.request

# ComfyUI başlatma parametreleri.
# --highvram KULLANILMAZ: modelleri VRAM'de kilitler ve LoRA/model geçişlerinde OOM'a yol açar.
import os as _os
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

COMFY_ARGS = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--preview-method", "none",
    "--reserve-vram", "2.0",
]


# ComfyUI çalışıyor mu kontrol et
comfy_running = False
try:
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    with urllib.request.urlopen(req, timeout=1) as resp:
        if resp.status == 200:
            comfy_running = True
except Exception:
    comfy_running = False

if not comfy_running:
    print("ComfyUI arka planda başlatılıyor...")
    with open("/content/comfyui.log", "w") as f:
        subprocess.Popen(COMFY_ARGS, cwd="/content/ComfyUI", stdout=f, stderr=subprocess.STDOUT)
    for _ in range(25):
        time.sleep(1)
        try:
            req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
            with urllib.request.urlopen(req, timeout=1) as resp:
                if resp.status == 200:
                    comfy_running = True
                    break
        except Exception:
            pass

# Faz-01 testini çalıştır
%cd /content/OzzyVision-Lab
!python tests/test_phase01_fidelity.py


## 6. Adım: 🚀 TEK TIKLA STÜDYO BAŞLATICI (Otomatik Hazır Olma Kontrollü)
ComfyUI headless, FastAPI backend ve Ngrok tünelini çalıştırır. ComfyUI aktif olana kadar otomatik olarak bekler ve arayüzü sunar.


In [ ]:
import os, time, subprocess, urllib.request, json
from pyngrok import ngrok
from IPython.display import display, HTML

# ComfyUI başlatma parametreleri.
# --highvram KULLANILMAZ: modelleri VRAM'de kilitler ve LoRA/model geçişlerinde OOM'a yol açar.
import os as _os
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

COMFY_ARGS = [
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--preview-method", "none",
    "--reserve-vram", "2.0",
]


# ==========================================
# NGROK VE PORT AYARLARI (Colab Secrets)
# ==========================================
NGROK_AUTH_TOKEN = ""
try:
    from google.colab import userdata
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTHTOKEN') or userdata.get('NGROK_AUTH_TOKEN') or ""
except Exception:
    NGROK_AUTH_TOKEN = os.getenv('NGROK_AUTHTOKEN') or os.getenv('NGROK_AUTH_TOKEN', "")

NGROK_DOMAIN = ""
try:
    from google.colab import userdata
    NGROK_DOMAIN = userdata.get('NGROK_DOMAIN') or ""
except Exception:
    NGROK_DOMAIN = os.getenv('NGROK_DOMAIN', "")

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("🔑 Colab Secrets: NGROK_AUTHTOKEN başarıyla yüklendi ve bağlandı.")
else:
    print("⚠️ UYARI: NGROK_AUTHTOKEN Colab Secrets içinde bulunamadı!")

# 1. ComfyUI Başlat & Dinle
comfy_ready = False
try:
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    with urllib.request.urlopen(req, timeout=1) as resp:
        if resp.status == 200:
            comfy_ready = True
except Exception:
    pass

if not comfy_ready:
    print("1. ComfyUI Headless başlatılıyor...")
    comfy_log = open("/content/comfyui.log", "w")
    subprocess.Popen(COMFY_ARGS, cwd="/content/ComfyUI", stdout=comfy_log, stderr=subprocess.STDOUT)
    
    print("   ⏳ ComfyUI portunun hazır olması bekleniyor...")
    for i in range(25):
        time.sleep(1)
        try:
            req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
            with urllib.request.urlopen(req, timeout=1) as resp:
                if resp.status == 200:
                    comfy_ready = True
                    print("   ✅ ComfyUI başarıyla aktif oldu!")
                    break
        except Exception:
            pass

if not comfy_ready:
    print("\n⚠️ UYARI: ComfyUI henüz yanıt vermiyor. Son log satırları:")
    !tail -n 12 /content/comfyui.log

# MiniMax H3 SaveVideo oto-onarım kontrolü
wf_f = '/content/OzzyVision-Lab/app/backend/engines/minimax_h3/ref2va_workflow.json'
if os.path.exists(wf_f):
    try:
        with open(wf_f, 'r', encoding='utf-8') as _rf:
            _wfd = json.load(_rf)
        if '17' in _wfd and 'inputs' in _wfd['17'] and 'format' not in _wfd['17']['inputs']:
            _wfd['17']['inputs']['format'] = 'auto'
            with open(wf_f, 'w', encoding='utf-8') as _wf:
                json.dump(_wfd, _wf, indent=2)
            print('🔧 ref2va_workflow.json SaveVideo format="auto" otomatik tamamlandı.')
    except Exception:
        pass

# 2. FastAPI Backend Başlat
!pkill -9 -f uvicorn 2>/dev/null || true
time.sleep(1)
print("2. FastAPI Backend başlatılıyor...")
backend_log = open("/content/backend.log", "w")
subprocess.Popen(
    ["uvicorn", "app.backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/OzzyVision-Lab",
    stdout=backend_log,
    stderr=subprocess.STDOUT
)

# Backend gerçekten ayağa kalkana kadar bekle (ngrok'u boş porta açmayı önler)
backend_ready = False
for _ in range(40):
    time.sleep(1)
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/status", timeout=2) as _r:
            if _r.status == 200:
                backend_ready = True
                print("   ✅ FastAPI backend aktif.")
                break
    except Exception:
        pass

if not backend_ready:
    print("   ❌ Backend başlatılamadı! Son log satırları:")
    !tail -n 25 /content/backend.log

# 3. Ngrok Tüneli Aç (FastAPI Portu 8000)
print("3. Ngrok public tüneli açılıyor...")
ngrok.kill()
tunnel_options = {"addr": 8000}
if NGROK_DOMAIN:
    tunnel_options["domain"] = NGROK_DOMAIN

public_url = ngrok.connect(**tunnel_options).public_url

# 4. Şık HTML Gösterge Kartı
display(HTML(f"""
<div style="background: linear-gradient(135deg, #090a0f 0%, #161a29 100%); border: 1px solid #6366f1; border-radius: 16px; padding: 24px; color: #fff; font-family: sans-serif; max-width: 650px; box-shadow: 0 8px 32px rgba(0,0,0,0.5);">
  <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 16px;">
    <div style="background: #6366f1; width: 40px; height: 40px; border-radius: 10px; display: flex; align-items: center; justify-content: center; font-size: 20px;">🎬</div>
    <div>
      <h2 style="margin: 0; font-size: 20px; font-weight: 800;">OzzyVision-Lab hazır</h2>
      <p style="margin: 2px 0 0; font-size: 12px; color: #94a3b8;">LTX-2.5 &amp; MiniMax H3 • Google Drive kalıcı depolama</p>
    </div>
  </div>
  <div style="background: rgba(255,255,255,0.05); padding: 16px; border-radius: 10px; margin-bottom: 16px;">
    <p style="margin: 0 0 8px; font-size: 13px; font-weight: 600;">🌐 Web Stüdyosu Adresi:</p>
    <a href="{public_url}" target="_blank" style="color: #38bdf8; font-size: 15px; font-weight: 700; text-decoration: none;">{public_url} &rarr;</a>
  </div>
  <div style="background: rgba(168,85,247,0.1); border: 1px solid rgba(168,85,247,0.3); padding: 16px; border-radius: 10px;">
    <p style="margin: 0 0 6px; font-size: 13px; font-weight: 600; color: #e9d5ff;">🤖 Claude Remote MCP Sunucu Adresi:</p>
    <code style="color: #f472b6; font-size: 13px;">{public_url}/mcp</code>
    <p style="margin: 6px 0 0; font-size: 11px; color: #cbd5e1;">Claude'da Settings &gt; Connectors &gt; Add Custom Connector diyerek bu URL'i ekleyin.</p>
  </div>
</div>
"""))
